# College Board Table Extractor for each state

In [14]:
import pdfplumber
import pandas as pd
import os
import re

#pdf_dir = './state_pdfs'
pdf_dir = './2019state_pdfs'
all_state_data = []

def clean_val(val):
    if val is None: return 0
    s_val = str(val).strip().split(' ')[0]
    clean_s = re.sub(r'[^\d.]', '', s_val)
    try:
        return float(clean_s) if '.' in clean_s else int(clean_s)
    except:
        return 0

def parse_full_row(line):
    clean_line = re.sub(r'\s+', ' ', line).replace(',', '').strip()
    parts = clean_line.split(' ')
    for i, part in enumerate(parts):
        if '%' in part:
            try:
                n_takers = float(parts[i-1])
                pct_takers = float(part.replace('%', '')) / 100.0
                mean_tot = float(parts[i+1])
                mean_erw = float(parts[i+2])
                mean_math = float(parts[i+3])
                met_both = float(parts[i+4].replace('%', '')) / 100.0
                met_erw = float(parts[i+5].replace('%', '')) / 100.0
                met_math = float(parts[i+6].replace('%', '')) / 100.0
                return n_takers, pct_takers, mean_tot, mean_erw, mean_math, met_both, met_erw, met_math
            except (ValueError, IndexError):
                continue
    return None

def extract_percentile_table(pdf):
    # PRE-INITIALIZE all 24 columns with 0
    cols = ['Total', 'ERW', 'Math', 'Reading', 'Writing', 'Math_Sub', 'Hist_Soc', 'Science']
    results = {}
    for label in ['75th', '50th', '25th']:
        for col in cols:
            results[f'SAT_{label}_{col}'] = 0

    last_page = pdf.pages[-1]
    text = last_page.extract_text()
    if not text: return results

    for label in ['75th', '50th', '25th']:
        # Look for the line starting with the label and grab all trailing numbers
        line_match = re.search(rf"{label}\s+([\d\s\.]+)", text)
        if line_match:
            raw_vals = re.findall(r'[\d\.]+', line_match.group(1))
            nums = [clean_val(v) for v in raw_vals]
            # Map found numbers to the pre-initialized keys
            for i, val in enumerate(nums):
                if i < len(cols):
                    results[f'SAT_{label}_{cols[i]}'] = val
    return results

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    # state_name = file_name.replace('2024-', '').split('-')[0].replace('_', ' ').title()
    
    # Split on '-' once to isolate year and state
    year, state_raw = file_name.split('-', 1)[0], file_name.split('-', 2)[1]
    
    # Clean state name
    state_name = state_raw.replace('_', ' ').title()

    if "Puerto" in state_name: state_name = "Puerto Rico"
    if "Virgin" in state_name: state_name = "US Virgin Islands"
    
    data = {'Year': year, 'State': state_name}
    sections = {
        'Race / Ethnicity': {'American Indian': 'Race_AmInd', 'Asian': 'Race_Asian', 'Black': 'Race_Black', 'Hispanic': 'Race_Hispanic', 'Native Hawaiian': 'Race_NatHaw', 'White': 'Race_White', 'Two or More': 'Race_Multi', 'No Response': 'Race_NoResp'},
        'Gender': {'Female': 'Gen_Female', 'Male': 'Gen_Male', 'No Response': 'Gen_NoResp'},
        'Highest Level of Parental Education': {'No High School Diploma': 'Edu_NoHS', 'High School Diploma': 'Edu_HS', 'Associate Degree': 'Edu_Assoc', 'Bachelor\'s Degree': 'Edu_Bach', 'Graduate Degree': 'Edu_Grad', 'No Response': 'Edu_NoResp'},
        'Family Income': {'Lowest Quintile': 'Inc_Lowest', '2nd Lowest Quintile': 'Inc_2ndLow', 'Middle Quintile': 'Inc_Middle', '2nd Highest Quintile': 'Inc_2ndHigh', 'Highest Quintile': 'Inc_Highest', 'Unknown': 'Inc_Unknown'},
        'School Location': {'City': 'Loc_City', 'Suburb': 'Loc_Suburb', 'Town/Rural': 'Loc_TownRural', 'Unknown': 'Loc_Unknown'}
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Stats (First 4 pages)
            for p_idx in range(min(4, len(pdf.pages))):
                p_text = pdf.pages[p_idx].extract_text() or ""
                t_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p_text, re.IGNORECASE)
                if t_m and not data.get('Total_SAT_Takers'): data['Total_SAT_Takers'] = clean_val(t_m.group(1))
                p_m = re.search(r'Participation\s+Rate\s+(\d+%)', p_text, re.IGNORECASE)
                if p_m and not data.get('Total_Participation_Rate'): data['Total_Participation_Rate'] = float(p_m.group(1).replace('%',''))/100

            # Means (Keyword search across all pages)
            for page in pdf.pages:
                text = page.extract_text() or ""
                if "Mean Score" in text:
                    nums = [int(n) for n in re.findall(r'\d{3,4}', text) if 300 < int(n) < 1600]
                    if len(nums) >= 3:
                        data['Mean_Total_Score'], data['Mean_ERW_Score'], data['Mean_Math_Score'] = nums[:3]
                        break

            # Demographics (Scan middle pages)
            current_sec_key = None
            for page in pdf.pages[1:10]: 
                text = page.extract_text()
                if not text: continue
                for line in text.split('\n'):
                    for sec_header in sections.keys():
                        if sec_header in line: current_sec_key = sec_header
                    if current_sec_key:
                        for label, prefix in sections[current_sec_key].items():
                            if label in line:
                                stats = parse_full_row(line)
                                if stats:
                                    (data[f'{prefix}_N'], data[f'{prefix}_Pct'], data[f'{prefix}_Mean_Tot'], 
                                     data[f'{prefix}_Mean_ERW'], data[f'{prefix}_Mean_Math'],
                                     data[f'{prefix}_Met_Both'], data[f'{prefix}_Met_ERW'], data[f'{prefix}_Met_Math']) = stats

            # Percentiles (Adaptive & Pre-initialized)
            data.update(extract_percentile_table(pdf))

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        print(f"Parsing: {filename}")
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data).fillna(0)

# Final calculation
if 'SAT_75th_Total' in df.columns and 'SAT_25th_Total' in df.columns:
    df['SAT_IQR_Spread'] = df['SAT_75th_Total'] - df['SAT_25th_Total']

df.to_csv('sat_comprehensive_final_2019.csv', index=False)
print(f"Process Complete. Master file saved with {len(df.columns)} columns.")

Parsing: 2019-alabama-sat-suite-assessments-annual-report.pdf
Parsing: 2019-alaska-sat-suite-assessments-annual-report.pdf
Parsing: 2019-arizona-sat-suite-assessments-annual-report.pdf
Parsing: 2019-arkansas-sat-suite-assessments-annual-report.pdf
Parsing: 2019-california-sat-suite-assessments-annual-report.pdf
Parsing: 2019-colorado-sat-suite-assessments-annual-report.pdf
Parsing: 2019-connecticut-sat-suite-assessments-annual-report.pdf
Parsing: 2019-delaware-sat-suite-assessments-annual-report.pdf
Parsing: 2019-district-columbia-sat-suite-assessments-annual-report.pdf
Parsing: 2019-florida-sat-suite-assessments-annual-report.pdf
Parsing: 2019-georgia-sat-suite-assessments-annual-report.pdf
Parsing: 2019-hawaii-sat-suite-assessments-annual-report.pdf
Parsing: 2019-idaho-sat-suite-assessments-annual-report.pdf
Parsing: 2019-illinois-sat-suite-assessments-annual-report.pdf
Parsing: 2019-indiana-sat-suite-assessments-annual-report.pdf
Parsing: 2019-iowa-sat-suite-assessments-annual-repor

In [15]:
import pdfplumber
import os
import re
import pandas as pd

def extract_top_sat_iqr(folder_path):
    results = []
    # Regex to find lines starting with the target percentiles
    pattern = re.compile(r'^(75th|50th|25th)\s+([\d\.]+.*)')

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            path = os.path.join(folder_path, filename)
            
            # Split on '-' once to isolate year and state
            year, state_raw = filename.split('-', 1)[0], filename.split('-', 2)[1]
            # Clean state name
            state_name = state_raw.replace('_', ' ').title()
            
            with pdfplumber.open(path) as pdf:
                page = pdf.pages[-1]
                text = page.extract_text()
                
                if text:
                    found_for_this_file = 0
                    for line in text.split('\n'):
                        line = line.strip()
                        match = pattern.match(line)
                        
                        # Only grab the first 3 matches (which are the SAT rows)
                        if match and found_for_this_file < 3:
                            percentile = match.group(1)
                            values = match.group(2).split()
                            #results.append([filename, percentile] + values)
                            results.append([year, state_name, percentile] + values)
                            found_for_this_file += 1
                        
                        # Stop searching this file once we have our 3 rows
                        if found_for_this_file == 3:
                            break

    columns = ["Year", "State", "Percentile", "Total", "ERW", "Math", "Reading", "Writing", "Math_Test", "Hist", "Science"]
    df = pd.DataFrame(results, columns=columns)
    return df

# Run and Save
folder = '2019state_pdfs'
sat_only_df = extract_top_sat_iqr(folder)
print(sat_only_df)
sat_only_df.to_csv("sat_only_iqr_2019.csv", index=False)

     Year      State Percentile Total  ERW Math Reading Writing Math_Test  \
0    2019    Alabama       75th  1290  660  640      33      33        32   
1    2019    Alabama       50th  1130  580  550      29      30      27.5   
2    2019    Alabama       25th   990  510  480      25      26        24   
3    2019     Alaska       75th  1220  620  600      31      31        30   
4    2019     Alaska       50th  1090  560  530      28      27      26.5   
..    ...        ...        ...   ...  ...  ...     ...     ...       ...   
154  2019  Wisconsin       50th  1300  640  660      32      33        33   
155  2019  Wisconsin       25th  1150  570  570      28      29      28.5   
156  2019    Wyoming       75th  1390  700  680      34      35        34   
157  2019    Wyoming       50th  1260  630  610      32      32      30.5   
158  2019    Wyoming       25th  1080  550  530      28      27      26.5   

    Hist Science  
0     33      33  
1     29      29  
2     25      25  

In [4]:
import pdfplumber
import os
import re
import pandas as pd

def extract_top_sat_iqr(folder_path):
    results = []

    # Regex to find lines starting with the target percentiles
    pattern = re.compile(r'^(75th|50th|25th)\s+([\d\.]+.*)')

    for filename in os.listdir(folder_path):
        if filename.endswith(".pdf"):
            path = os.path.join(folder_path, filename)

            # Extract year and state
            year = filename.split('-', 1)[0]
            state_raw = filename.split('-', 2)[1]
            state_name = state_raw.replace('_', ' ').title()

            # Prepare a row dict for this PDF
            row = {
                "Year": year,
                "State": state_name
            }

            with pdfplumber.open(path) as pdf:
                page = pdf.pages[-1]
                text = page.extract_text()

                if text:
                    found = 0
                    for line in text.split('\n'):
                        line = line.strip()
                        match = pattern.match(line)

                        if match and found < 3:
                            percentile = match.group(1)
                            values = match.group(2).split()

                            # Expected order of extracted values
                            colnames = ["Total", "ERW", "Math", "Reading", "Writing",
                                        "Math_Test", "Hist", "Science"]

                            # Build prefixed columns
                            for col, val in zip(colnames, values):
                                row[f"{percentile}_{col}"] = val

                            found += 1

                        if found == 3:
                            break

            # Append the single flattened row for this PDF
            results.append(row)

    # Build DataFrame from list of dicts
    df = pd.DataFrame(results)
    return df


# Run and Save
folder = '2022state_pdfs'
sat_only_df = extract_top_sat_iqr(folder)
print(sat_only_df)
sat_only_df.to_csv("sat_only_iqr_2022.csv", index=False)


    Year          State 75th_Total 75th_ERW 75th_Math 75th_Reading  \
0   2022        Alabama       1300      660       650           33   
1   2022         Alaska       1230      630       600           32   
2   2022        Arizona       1300      660       650           33   
3   2022       Arkansas       1330      680       660           34   
4   2022     California       1310      650       660           33   
5   2022       Colorado       1160      590       570           30   
6   2022    Connecticut       1180      600       580           30   
7   2022       Delaware       1090      560       540           28   
8   2022       District       1180      600       580           30   
9   2022        Florida       1120      580       540           29   
10  2022        Georgia       1190      610       580           31   
11  2022         Hawaii       1250      630       620           32   
12  2022          Idaho       1110      570       550           29   
13  2022       Illin

In [ ]:
import pdfplumber
import pandas as pd
import os
import re

#pdf_dir = './state_pdfs'
pdf_dir = './2022state_pdfs'
all_state_data = []

def clean_val(val):
    if val is None: return 0
    s_val = str(val).strip().split(' ')[0]
    clean_s = re.sub(r'[^\d.]', '', s_val)
    try:
        return float(clean_s) if '.' in clean_s else int(clean_s)
    except:
        return 0

def parse_full_row(line):
    clean_line = re.sub(r'\s+', ' ', line).replace(',', '').strip()
    parts = clean_line.split(' ')
    for i, part in enumerate(parts):
        if '%' in part:
            try:
                n_takers = float(parts[i-1])
                pct_takers = float(part.replace('%', '')) / 100.0
                mean_tot = float(parts[i+1])
                mean_erw = float(parts[i+2])
                mean_math = float(parts[i+3])
                met_both = float(parts[i+4].replace('%', '')) / 100.0
                met_erw = float(parts[i+5].replace('%', '')) / 100.0
                met_math = float(parts[i+6].replace('%', '')) / 100.0
                return n_takers, pct_takers, mean_tot, mean_erw, mean_math, met_both, met_erw, met_math
            except (ValueError, IndexError):
                continue
    return None

def extract_percentile_table(pdf):
    # PRE-INITIALIZE all 24 columns with 0
    cols = ['Total', 'ERW', 'Math', 'Reading', 'Writing', 'Math_Sub', 'Hist_Soc', 'Science']
    results = {}
    for label in ['75th', '50th', '25th']:
        for col in cols:
            results[f'SAT_{label}_{col}'] = 0

    last_page = pdf.pages[-1]
    text = last_page.extract_text()
    if not text: return results

    for label in ['75th', '50th', '25th']:
        # Look for the line starting with the label and grab all trailing numbers
        line_match = re.search(rf"{label}\s+([\d\s\.]+)", text)
        if line_match:
            raw_vals = re.findall(r'[\d\.]+', line_match.group(1))
            nums = [clean_val(v) for v in raw_vals]
            # Map found numbers to the pre-initialized keys
            for i, val in enumerate(nums):
                if i < len(cols):
                    results[f'SAT_{label}_{cols[i]}'] = val
    return results

def extract_top_sat_iqr(pdf_path):
    # Regex to find lines starting with the target percentiles
    pattern = re.compile(r'^(75th|50th|25th)\s+([\d\.]+.*)')

    
    # Prepare a row dict for this PDF
    row = {}
    with pdfplumber.open(pdf_path) as pdf:
        page = pdf.pages[-1]
        text = page.extract_text()

        if text:
            found = 0
            for line in text.split('\n'):
                line = line.strip()
                match = pattern.match(line)

                if match and found < 3:
                    percentile = match.group(1)
                    values = match.group(2).split()

                    # Expected order of extracted values
                    colnames = ["Total", "ERW", "Math", "Reading", "Writing",
                                "Math_Test", "Hist", "Science"]

                    # Build prefixed columns
                    for col, val in zip(colnames, values):
                        row[f"{percentile}_{col}"] = val

                    found += 1

                if found == 3:
                    break

    return row

def extract_state_data(pdf_path):
    file_name = os.path.basename(pdf_path)
    # state_name = file_name.replace('2024-', '').split('-')[0].replace('_', ' ').title()
    
    # Split on '-' once to isolate year and state
    year, state_raw = file_name.split('-', 1)[0], file_name.split('-', 2)[1]
    
    # Clean state name
    state_name = state_raw.replace('_', ' ').title()

    if "Puerto" in state_name: state_name = "Puerto Rico"
    if "Virgin" in state_name: state_name = "US Virgin Islands"
    
    data = {'Year': year, 'State': state_name}
    sections = {
        'Race / Ethnicity': {'American Indian': 'Race_AmInd', 'Asian': 'Race_Asian', 'Black': 'Race_Black', 'Hispanic': 'Race_Hispanic', 'Native Hawaiian': 'Race_NatHaw', 'White': 'Race_White', 'Two or More': 'Race_Multi', 'No Response': 'Race_NoResp'},
        'Gender': {'Female': 'Gen_Female', 'Male': 'Gen_Male', 'No Response': 'Gen_NoResp'},
        'Highest Level of Parental Education': {'No High School Diploma': 'Edu_NoHS', 'High School Diploma': 'Edu_HS', 'Associate Degree': 'Edu_Assoc', 'Bachelor\'s Degree': 'Edu_Bach', 'Graduate Degree': 'Edu_Grad', 'No Response': 'Edu_NoResp'},
        'Family Income': {'Lowest Quintile': 'Inc_Lowest', '2nd Lowest Quintile': 'Inc_2ndLow', 'Middle Quintile': 'Inc_Middle', '2nd Highest Quintile': 'Inc_2ndHigh', 'Highest Quintile': 'Inc_Highest', 'Unknown': 'Inc_Unknown'},
        'School Location': {'City': 'Loc_City', 'Suburb': 'Loc_Suburb', 'Town/Rural': 'Loc_TownRural', 'Unknown': 'Loc_Unknown'}
    }

    try:
        with pdfplumber.open(pdf_path) as pdf:
            # Stats (First 4 pages)
            for p_idx in range(min(4, len(pdf.pages))):
                p_text = pdf.pages[p_idx].extract_text() or ""
                t_m = re.search(r'SAT\s+Takers\D+([\d,]+)', p_text, re.IGNORECASE)
                if t_m and not data.get('Total_SAT_Takers'): data['Total_SAT_Takers'] = clean_val(t_m.group(1))
                p_m = re.search(r'Participation\s+Rate\s+(\d+%)', p_text, re.IGNORECASE)
                if p_m and not data.get('Total_Participation_Rate'): data['Total_Participation_Rate'] = float(p_m.group(1).replace('%',''))/100

            # Means (Keyword search across all pages)
            for page in pdf.pages:
                text = page.extract_text() or ""
                if "Mean Score" in text:
                    nums = [int(n) for n in re.findall(r'\d{3,4}', text) if 300 < int(n) < 1600]
                    if len(nums) >= 3:
                        data['Mean_Total_Score'], data['Mean_ERW_Score'], data['Mean_Math_Score'] = nums[:3]
                        break

            # Demographics (Scan middle pages)
            current_sec_key = None
            for page in pdf.pages[1:10]: 
                text = page.extract_text()
                if not text: continue
                for line in text.split('\n'):
                    for sec_header in sections.keys():
                        if sec_header in line: current_sec_key = sec_header
                    if current_sec_key:
                        for label, prefix in sections[current_sec_key].items():
                            if label in line:
                                stats = parse_full_row(line)
                                if stats:
                                    (data[f'{prefix}_N'], data[f'{prefix}_Pct'], data[f'{prefix}_Mean_Tot'], 
                                     data[f'{prefix}_Mean_ERW'], data[f'{prefix}_Mean_Math'],
                                     data[f'{prefix}_Met_Both'], data[f'{prefix}_Met_ERW'], data[f'{prefix}_Met_Math']) = stats

            # Percentiles (Adaptive & Pre-initialized)
            #data.update(extract_percentile_table(pdf))
            data.update(extract_top_sat_iqr(pdf_path))

    except Exception as e:
        print(f"Error in {state_name}: {e}")
            
    return data

# --- Execution ---
for filename in os.listdir(pdf_dir):
    if filename.lower().endswith(".pdf"):
        print(f"Parsing: {filename}")
        all_state_data.append(extract_state_data(os.path.join(pdf_dir, filename)))

df = pd.DataFrame(all_state_data).fillna(0)

# Final calculation
#if 'SAT_75th_Total' in df.columns and 'SAT_25th_Total' in df.columns:
#    df['SAT_IQR_Spread'] = df['SAT_75th_Total'] - df['SAT_25th_Total']

df.to_csv('sat_comprehensive_final_2021.csv', index=False)
print(f"Process Complete. Master file saved with {len(df.columns)} columns.")

Parsing: 2021-alabama-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-alaska-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-arizona-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-arkansas-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-california-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-colorado-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-connecticut-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-delaware-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-district-of-columbia-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-florida-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-georgia-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-hawaii-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-idaho-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-illinois-sat-suite-of-assessments-annual-report.pdf
Parsing: 2021-indiana-sat-suite-of-assessments-annual-report.pdf
Parsi